In [ ]:
# Setup (added for the release): the notebook's folder on sys.path, then work in its data folder.
# The curation ran with the current directory = that folder; see curation/README.md for its contents.
import os, sys
if "data/curation/" not in os.getcwd().replace(os.sep, "/") + "/":
    sys.path.insert(0, os.path.abspath("."))
    os.chdir('../../data/curation/rBMRB')
print("working folder:", os.getcwd())


# Old split

Finds the entries kept by both PANAV and LACS and makes the old split. `Screening_7_match.ipynb` makes the released split from it.

## 1. Entries kept by both PANAV and LACS

In [ ]:
import json
from collections import Counter

# ===== 1. read the JSON files =====
with open("bmrb_cs_data_withoffset_panav_v1.json") as f:
    A = json.load(f)

with open("bmrb_cs_data_withoffset_lacs_v1.json") as f:
    B = json.load(f)

# ===== 2. reverse index directly on entry_ID =====
# [FIX] the earlier version went through the sequence: seq_to_keyB maps one sequence to one
# keyB, so different entry_IDs sharing a sequence were silently overwritten (325
# keyBs were lost and 308 pairs that should match did not). Since the goal is
# to match identical entry_IDs, a reverse index on entry_ID and the intersection suffice, without the sequence.
def build_entry_id_index(data, label):
    counts = Counter(data["entry_ID"].values())
    dups = {eid: c for eid, c in counts.items() if c > 1}
    if dups:
        print(f"WARNING: {label} has {len(dups)} duplicated entry_ID values (kept the last key seen): {list(dups)[:5]}")
    return {entry_id: key for key, entry_id in data["entry_ID"].items()}

entry_id_to_keyA = build_entry_id_index(A, "A (panav)")
entry_id_to_keyB = build_entry_id_index(B, "B (lacs)")

# ===== 3. records with the same entry_ID on both sides =====
matches = {
    keyA: entry_id_to_keyB[entry_id]
    for entry_id, keyA in entry_id_to_keyA.items()
    if entry_id in entry_id_to_keyB
}

# ===== 4. save =====
with open("matched_keys_LACS_PANAV.json", "w") as f:
    json.dump(matches, f, indent=4)

print("Matches found:", len(matches))

In [ ]:
import json
# LACS keys of matched_keys_LACS_PANAV.json that share a value
with open("matched_keys_LACS_PANAV.json", "r") as f:
    match_panav_keys = json.load(f)

with open("bmrb_cs_data_withoffset_panav_v1.json", "r") as f:
    bmrb_panav = json.load(f)

# group by PANAV key
value_to_panav_keys = {}
for panav_key, lacs_key in match_panav_keys.items():
    value_to_panav_keys.setdefault(lacs_key, []).append(panav_key)

# keep only groups with a repeated value and print their sequence
# Write output into a file
for lacs_key, panav_keys in value_to_panav_keys.items():
    if len(panav_keys) > 1:
        print(f"LACS value: {lacs_key}")
        for panav_key in panav_keys:
            seq = bmrb_panav["entry_ID"].get(panav_key)
            print(f"  {panav_key}: {seq}")
        print("-" * 60 + "\n")

In [ ]:
import json

with open("bmrb_cs_data_withoffset_panav_v1.json") as f:
    A = json.load(f)

with open("bmrb_cs_data_withoffset_lacs_v1.json") as f:
    B = json.load(f)

with open("matched_keys_LACS_PANAV.json") as f:
    matches = json.load(f)

seq_mismatches = []
for keyA, keyB in matches.items():
    seqA = A["sequence"].get(keyA)
    seqB = B["sequence"].get(keyB)
    if seqA != seqB:
        seq_mismatches.append((keyA, keyB, A["entry_ID"].get(keyA), seqA, seqB))

print(f"Checked {len(matches)} matched pairs.")
print(f"Sequence mismatches: {len(seq_mismatches)}")
for keyA, keyB, entry_id, seqA, seqB in seq_mismatches[:20]:
    print(f"\nentry_ID={entry_id}  (A key={keyA}, B key={keyB})")
    print(f"  A seq ({len(seqA) if seqA else 0} aa): {seqA}")
    print(f"  B seq ({len(seqB) if seqB else 0} aa): {seqB}")

In [ ]:
import json

with open("bmrb_cs_data_withoffset_panav_v1.json", 'r') as f:
    panav_data = json.load(f)

with open("matched_keys_LACS_PANAV.json", 'r') as f:
    match_panav_keys = json.load(f)

for key in list(panav_data["sequence"].keys()):
    if key not in match_panav_keys:
        for big_key in panav_data.keys():
            del panav_data[big_key][key]  # Remove the entry with flag for now, can be further processed later
    
with open('bmrb_cs_data_withoffset_panav_v2.json', 'w') as f:
    json.dump(panav_data, f, indent=4)
print(len(panav_data['sequence']))

In [ ]:
import json

with open("bmrb_cs_data_withoffset_lacs_v1.json", 'r') as f:
    lacs_data = json.load(f)

with open("matched_keys_LACS_PANAV.json", 'r') as f:
    match_panav_keys = json.load(f)

for key in list(lacs_data["sequence"].keys()):
    if key not in list(match_panav_keys.values()):  # Check against the values of the match_panav_keys dictionary
        for big_key in lacs_data.keys():
            del lacs_data[big_key][key]  # Remove the entry with flag for now, can be further processed later

with open('bmrb_cs_data_withoffset_lacs_v2.json', 'w') as f:
    json.dump(lacs_data, f, indent=4)
print(len(lacs_data['sequence']))

## 2. Structures of these entries

`LACS_PANAV_structures_v2/` is read by `Screening_7_match.ipynb`.

In [ ]:
from pathlib import Path
import shutil
import json

SRC_DIR = Path("../rBMRB/LACS_PANAV_structures")
DST_DIR = Path("../rBMRB/LACS_PANAV_structures_v2")

DST_DIR.mkdir(parents=True, exist_ok=True)

def bare(stem):
    """Strip the test_ prefix added for forced-test entries."""
    return stem[len("test_"):] if stem.startswith("test_") else stem

# Map bare entry id -> file path in the old structure dir (file names may carry the test_ prefix)
src_lookup = {}
for src_path in SRC_DIR.glob("*.pdb"):
    src_lookup.setdefault(bare(src_path.stem), []).append(src_path)

# Entry set of the new database (LACS ∩ new PANAV)
with open("bmrb_cs_data_withoffset_lacs_v2.json", 'r') as f:
    data = json.load(f)
    entries = {entry.removeprefix("bmr") for entry in data['entry_ID'].values()}

missing, copied, conflicts = [], [], []
for entry in sorted(entries):
    matches = src_lookup.get(entry)
    if not matches:
        missing.append(entry)
        continue
    if len(matches) > 1:
        conflicts.append((entry, [p.name for p in matches]))
    src_path = matches[0]
    shutil.copy2(src_path, DST_DIR / src_path.name)   # keep original file name, incl. test_ prefix
    copied.append(src_path.name)

dropped = sorted(set(src_lookup) - entries)

print(f"Copied {len(copied)} files to {DST_DIR}")
print(f"{len(dropped)} entries in SRC_DIR not in the new database (not copied): {dropped}")
if missing:
    print(f"{len(missing)} entries not found in SRC_DIR:")
    for m in missing:
        print(" ", m)
if conflicts:
    print(f"{len(conflicts)} entries had multiple files (only first copied):")
    for e, names in conflicts:
        print(" ", e, names)

## 3. Old split

`cluster_30p_old/result_cluster.tsv` is in the curation inputs. It was made in `cluster_30p_old/` with:

```
foldseek easy-cluster ../LACS_PANAV_structures result tmp --alignment-type 1 --tmscore-threshold 0.3 --min-seq-id 0.3 --cov-mode 0 --threads 8
```

`Screening_7_match.ipynb` starts from the split written by the cell below.

In [ ]:
"""
Split the foldseek clusters (col1 = cluster_id, col2 = entry_id) into
train / val / test. Rule: only a cluster containing an entry_id (test_ prefix removed) of the
FORCED_TEST_IDS list is forced into test; other entries are not forced even with the test_ prefix.
test / val are filled up to N_TEST / N_VAL clusters; all other clusters go to train.
"""
import random
from collections import defaultdict
from pathlib import Path

TSV_PATH = Path("cluster_30p_old/result_cluster.tsv")
OUT_DIR = Path("cluster_30p_old/split_output")
N_TEST = 250      # target number of test clusters
N_VAL = 300      # target number of val clusters
SEED = 42

FORCED_TEST_IDS = {
    "11139_1", "36329_1", "30706_1", "6032_1", "16603_1", "10267_1", "6726_1", "52187_1", "19743_1", "11202_1", 
           "34106_1", "4463_1", "4819_1", "5921_1", "6933_1", "16628_1", "34043_1", "26517_1", "34425_1", "11188_1", 
           "19298_1", "25895_1", "52088_1", "16387_1", "5902_1", "6782_1", "10235_1", "19737_1", "17136_1", "34607_1", 
           "50270_1", "6114_1", "25968_1", "5842_1", "27687_1", "16016_1", "25448_1", "36327_1", "51126_1", "17723_1",
           "16848_1", "4671_1", "5490_1", "34678_1", "19231_1", "18145_1", "16652_1", "16186_1"
}

OUT_DIR.mkdir(parents=True, exist_ok=True)

def strip_test_prefix(entry_id):
    return entry_id[len("test_"):] if entry_id.startswith("test_") else entry_id

# 1. read and group by cluster_id
clusters = defaultdict(list)  # cluster_id -> [entry_id, ...]
with open(TSV_PATH) as f:
    for line in f:
        line = line.rstrip("\n")
        if not line:
            continue
        cluster_id, entry_id = line.split("\t")
        clusters[cluster_id].append(entry_id)

# 2. clusters with an entry of FORCED_TEST_IDS go to test first
random.seed(SEED)
forced_test_clusters, remaining_clusters = [], []
for cluster_id, entries in clusters.items():
    if any(strip_test_prefix(e) in FORCED_TEST_IDS for e in entries):
        forced_test_clusters.append(cluster_id)
    else:
        remaining_clusters.append(cluster_id)

remaining_clusters = sorted(remaining_clusters)  # fixed order (reproducible)
random.shuffle(remaining_clusters)

# 3. fill test up to N_TEST clusters
if len(forced_test_clusters) > N_TEST:
    raise ValueError(
        f"clusters with FORCED_TEST_IDS entries ({len(forced_test_clusters)}) exceed N_TEST={N_TEST}; "
        "cannot reduce further: raise N_TEST or check the data."
    )
n_extra_test = N_TEST - len(forced_test_clusters)
extra_test_clusters = remaining_clusters[:n_extra_test]
remaining_clusters = remaining_clusters[n_extra_test:]
test_clusters = forced_test_clusters + extra_test_clusters

# 4. N_VAL of the remaining clusters go to val, the rest to train
if len(remaining_clusters) < N_VAL:
    raise ValueError(
        f"remaining clusters ({len(remaining_clusters)}) are fewer than N_VAL={N_VAL}."
    )
val_clusters = remaining_clusters[:N_VAL]
train_clusters = remaining_clusters[N_VAL:]

print(f"clusters: {len(clusters)}")
print(f"test clusters: {len(test_clusters)} (forced: {len(forced_test_clusters)}, added: {len(extra_test_clusters)})")
print(f"val clusters: {len(val_clusters)}")
print(f"train clusters: {len(train_clusters)}")

def write_split(name, cluster_ids):
    entry_ids = sorted({strip_test_prefix(e) for cid in cluster_ids for e in clusters[cid]})
    out_path = OUT_DIR / f"{name}_entry_ids.txt"
    with open(out_path, "w") as f:
        f.write("\n".join(entry_ids) + "\n")
    print(f"{name}: {len(cluster_ids)} clusters, {len(entry_ids)} entries -> {out_path}")

write_split("test", test_clusters)
write_split("val", val_clusters)
write_split("train", train_clusters)